# Jon2 AI — Kaggle GPU Training

Kaggle-compatible private training workflow for Qwen3.5-2B. Keep this notebook and all attached datasets private. The training data must be attached as a **Private Kaggle Dataset** containing `source_approvals.json` and the exact approved `.txt` dialogue files. The notebook refuses to train if the approval manifest or CUDA GPU is missing. Training artifacts are written to `/kaggle/working` and become notebook outputs when you save a notebook version.

**Never make the dataset or notebook public.** Existing approvals allow private experimental research only.

In [ ]:
# 1) Clone the Jon2 training code
from pathlib import Path
import subprocess, sys, json, os
REPO = Path('/kaggle/working/jon2-ai')
if not REPO.exists():
    subprocess.run(['git','clone','--depth','1','--branch','feature/qwen25-arabic-lora-pipeline','https://github.com/jonkabeel-star/jon2-ai.git',str(REPO)],check=True)
os.chdir(REPO)
print('Repository:', REPO)


In [ ]:
# 2) Verify GPU and find the private, approved training dataset
import torch
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('GPU is not enabled/available. In Notebook Settings choose an available GPU accelerator; do not continue on CPU.')
print('GPU:', torch.cuda.get_device_name(0))
candidates = list(Path('/kaggle/input').rglob('source_approvals.json'))
valid = [p for p in candidates if list(p.parent.glob('*.txt'))]
if len(valid) != 1:
    raise RuntimeError(f'Expected exactly one attached private dataset folder with source_approvals.json and .txt files; found {len(valid)}. Attach the correct PRIVATE training-data dataset. Do not upload private training data to GitHub.')
DATA_DIR = valid[0].parent
APPROVAL = DATA_DIR / 'source_approvals.json'
OUTPUT = Path('/kaggle/working/jon2-qwen35-2b-lora')
if OUTPUT.exists() and any(OUTPUT.iterdir()):
    raise RuntimeError(f'Output folder already contains files: {OUTPUT}. Preserve previous artifacts; choose a new output path before retraining.')
print('Approved-data folder:', DATA_DIR)
print('Output folder:', OUTPUT)


In [ ]:
# 3) Install the project's Qwen3.5 / Unsloth training dependencies
subprocess.run([sys.executable,'-m','pip','install','-r','ai_engine/requirements-lora.txt'],check=True)
print('Dependencies installed. Restart the session only if Kaggle explicitly requires it; then rerun cells 2–4.')


In [ ]:
# 4) Rights, privacy, parser and split dry-run — no model weights are changed
subprocess.run([sys.executable,'ai_engine/finetune_qwen_lora.py','--data',str(DATA_DIR),'--approval-manifest',str(APPROVAL),'--output',str(OUTPUT),'--dry-run'],check=True)


## Training gate

Review the dry-run report. Proceed only if all source files match their approved SHA-256 hashes, privacy checks pass, and train/validation/test splits are non-empty and disjoint. If the GPU runs out of memory or any check fails, stop. Do not remove the rights checks or switch to CPU.

In [ ]:
# 5) REAL training. This is the only cell that performs optimizer steps.
cmd = [sys.executable,'ai_engine/finetune_qwen_lora.py','--data',str(DATA_DIR),'--approval-manifest',str(APPROVAL),'--output',str(OUTPUT),'--execute','--export-gguf']
subprocess.run(cmd,check=True)
metadata_file = OUTPUT / 'jon2_lora_training_metadata.json'
if not metadata_file.exists():
    raise RuntimeError('Training metadata is missing; training is not verified as complete.')
metadata = json.loads(metadata_file.read_text(encoding='utf-8'))
assert metadata.get('training_completed') is True
assert int(metadata.get('optimizer_steps_completed',0)) > 0
assert metadata.get('validation_evaluation') is not None
assert metadata.get('final_held_out_test_evaluation') is not None
print('VERIFIED: optimizer steps completed and evaluation metadata saved.')


In [ ]:
# 6) Inventory outputs. Save a private notebook version to retain these artifacts.
for p in sorted(OUTPUT.rglob('*')):
    if p.is_file(): print(f'{p} — {p.stat().st_size:,} bytes')
print('\nKeep notebook and attached datasets PRIVATE. Do not enable public chat from training success alone; run the held-out Arabic benchmark and human review first.')
